# Step 2: Baseline test of pretrained detectors (no training)

**Goal:** measure how good free pretrained detectors already are on three kinds of audio, before we train anything.

| Test set | Real | Fake |
|---|---|---|
| Hindi | Kathbath (Indian speakers) | IndicSynth (XTTS-v2 / VITS / FreeVC clones) |
| Commercial TTS | YouTube speech | ElevenLabs, Polly, Hume, Speechify, Kokoro… |
| In-the-Wild | Celebrity speech | Real deepfakes found online |

Models:
- **A:** NII AntiDeepfake wav2vec2-Large (ONNX export, so no fairseq needed)
- **B:** `garystafford/wav2vec2-deepfake-voice-detector` (what several other SIH teams use)

**Settings:** GPU T4 x2, Internet ON, secret `HF_TOKEN` attached. Run all cells top to bottom (about 20–30 min, mostly downloads).


In [ ]:
# CELL 1: install ONNX Runtime with GPU support
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu


In [ ]:
# CELL 2: setup + helpers
import os, zipfile, tempfile, random
import numpy as np, pandas as pd, librosa, torch
import pyarrow.parquet as pq
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download
from datasets import load_dataset, Audio
from sklearn.metrics import roc_auc_score, roc_curve

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
random.seed(0); np.random.seed(0)
N  = 300      # clips per class per test set
SR = 16000

def decode(audio_bytes, name_hint=""):
    """Any audio bytes -> 16 kHz mono float32."""
    ext = os.path.splitext(name_hint or "")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
        f.write(audio_bytes)
    try:
        wav, _ = librosa.load(f.name, sr=SR, mono=True)
    finally:
        os.remove(f.name)
    return wav.astype(np.float32)

def trim(wav):
    """Cut leading/trailing silence with the SAME rule for real and fake (avoids the silence shortcut)."""
    t, _ = librosa.effects.trim(wav, top_db=30)
    return t if len(t) > SR // 2 else wav

def eer_auc(labels, p_fake):
    """EER (%), AUC, best threshold, and accuracy if you naively use 0.5."""
    labels, p_fake = np.asarray(labels), np.asarray(p_fake)
    fpr, tpr, thr = roc_curve(labels, p_fake)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return {"EER_%": 100 * (fpr[i] + fnr[i]) / 2,
            "AUC": roc_auc_score(labels, p_fake),
            "threshold_at_EER": thr[i],
            "accuracy_at_0.5_%": 100 * ((p_fake >= 0.5).astype(int) == labels).mean()}

TEST = {}   # name -> (list of waveforms, list of labels); label 0 = real, 1 = fake
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - check settings")


In [ ]:
# CELL 3a: Hindi - real Kathbath vs fake IndicSynth
kb_path = hf_hub_download("ai4bharat/Kathbath", "hindi/valid-00000-of-00002.parquet", repo_type="dataset")
kb = pq.read_table(kb_path).to_pandas()
kb = kb.sample(frac=1, random_state=0).groupby("speaker_id").head(5).head(N)   # max 5 clips per speaker
real = [trim(decode(a["bytes"], a.get("path"))) for a in kb["audio_filepath"]]
print(f"Kathbath real: {len(real)} clips from {kb.speaker_id.nunique()} speakers")

isyn = load_dataset("vdivyasharma/IndicSynth", name="Hindi", split="train", streaming=True)
acol = next((k for k, v in (isyn.features or {}).items() if isinstance(v, Audio)), "audio")
isyn = isyn.cast_column(acol, Audio(decode=False)).shuffle(seed=0, buffer_size=2000)
fake, isyn_meta = [], []
for row in isyn:
    fake.append(trim(decode(row[acol]["bytes"], row[acol].get("path"))))
    isyn_meta.append({k: v for k, v in row.items() if k != acol})
    if len(fake) >= N:
        break
print(f"IndicSynth fake: {len(fake)} clips")
print("IndicSynth metadata example:", isyn_meta[0])
TEST["Hindi (Kathbath vs IndicSynth)"] = (real + fake, [0] * len(real) + [1] * len(fake))


In [ ]:
# CELL 3b: Modern commercial TTS (ElevenLabs, Polly, Hume, Speechify, Kokoro, Luvvoice) vs real speech
gs = load_dataset("garystafford/deepfake-audio-detection", split="train")
gs = gs.cast_column("audio", Audio(decode=False)).shuffle(seed=0)
wavs, labels = [], []
for lab in (0, 1):                       # this dataset: 0 = real, 1 = fake
    sub = gs.filter(lambda l: l == lab, input_columns="label").select(range(N))
    for a in sub["audio"]:
        wavs.append(trim(decode(a["bytes"], a.get("path"))))
        labels.append(lab)
TEST["Commercial TTS (garystafford)"] = (wavs, labels)
print("Commercial TTS clips:", len(wavs))


In [ ]:
# CELL 3c: In-the-Wild (real deepfakes of celebrities, English). 8 GB download, 5-10 min.
zp = hf_hub_download("mueller91/In-The-Wild", "release_in_the_wild.zip", repo_type="dataset")
zf = zipfile.ZipFile(zp)
meta_name = next(n for n in zf.namelist() if n.endswith("meta.csv"))
root = os.path.dirname(meta_name)
itw = pd.read_csv(zf.open(meta_name))
print(itw.head(3)); print(itw["label"].value_counts())
itw["y"] = itw["label"].astype(str).str.lower().str.contains("spoof").astype(int)
wavs, labels = [], []
for lab in (0, 1):
    for f in itw[itw.y == lab].sample(N, random_state=0)["file"]:
        wavs.append(trim(decode(zf.read(f"{root}/{f}" if root else f), f)))
        labels.append(lab)
TEST["In-the-Wild (English)"] = (wavs, labels)
print("In-the-Wild clips:", len(wavs))


In [ ]:
# CELL 4: Model A - NII AntiDeepfake wav2vec2-Large (ONNX)
import onnxruntime as ort
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()                     # lets ONNX Runtime find the CUDA libs that PyTorch ships
onnx_path = hf_hub_download("SpeechAntiSpoofingBenchmarks/Wav2Vec2-Large-AntiDeepfake",
                            "wav2vec2-large-antideepfake.onnx")
sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("Running on:", sess.get_providers()[0], "| input:", inp.name, inp.shape,
      "| output:", sess.get_outputs()[0].shape)
T = inp.shape[1] if isinstance(inp.shape[1], int) else 64600    # fixed window (~4 s)
print("Window length:", T, "samples =", round(T / SR, 2), "s")

def crop(wav, T=T):
    if len(wav) < T:
        wav = np.tile(wav, T // len(wav) + 1)      # repeat-pad short clips
    return wav[:T]

def score_A(wavs, bs=16):
    out = []
    for i in range(0, len(wavs), bs):
        x = np.stack([crop(w) for w in wavs[i:i + bs]])
        x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)   # per-clip layer norm
        logits = sess.run(None, {inp.name: x.astype(np.float32)})[0]
        if logits.shape[1] == 1:                 # single "real" logit
            out.extend(1 / (1 + np.exp(logits[:, 0])))
        else:                                    # [fake, real] per NII model card -> index 0 = fake
            e = np.exp(logits - logits.max(1, keepdims=True))
            out.extend((e / e.sum(1, keepdims=True))[:, 0])
    return np.array(out)

results = []
for name, (w, y) in TEST.items():
    results.append({"model": "A: AntiDeepfake W2V-Large", "test set": name, **eer_auc(y, score_A(w))})
    print(results[-1])


In [ ]:
# CELL 5: Model B - garystafford/wav2vec2-deepfake-voice-detector (used by other SIH teams)
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
mid = "garystafford/wav2vec2-deepfake-voice-detector"
fe = AutoFeatureExtractor.from_pretrained(mid)
mB = AutoModelForAudioClassification.from_pretrained(mid).cuda().eval()
print("Model B labels:", mB.config.id2label)
fake_idx = next((int(i) for i, l in mB.config.id2label.items()
                 if any(s in l.lower() for s in ("fake", "spoof", "synth"))), 1)
print("Using index", fake_idx, "as FAKE")

@torch.no_grad()
def score_B(wavs, bs=16):
    out = []
    for i in range(0, len(wavs), bs):
        batch = [crop(w) for w in wavs[i:i + bs]]
        x = fe(batch, sampling_rate=SR, return_tensors="pt", padding=True).input_values.cuda()
        out.extend(mB(x).logits.softmax(-1)[:, fake_idx].cpu().numpy())
    return np.array(out)

for name, (w, y) in TEST.items():
    results.append({"model": "B: garystafford wav2vec2", "test set": name, **eer_auc(y, score_B(w))})
    print(results[-1])


In [ ]:
# CELL 6: results table
df = pd.DataFrame(results).round(3)
display(df)
df.to_csv("/kaggle/working/step2_baseline_results.csv", index=False)
bad = df[df.AUC < 0.5]
if len(bad):
    print("WARNING: AUC < 0.5 means real/fake are flipped - label index likely reversed for:",
          bad[["model", "test set"]].values.tolist())
print("Done. Lower EER is better (0 = perfect, 50 = coin flip).")
